# GHOST IN THE DATA 🕵️
## AI-Powered Data Forensics & Anomaly Investigation

**Goal:** investigate a dataset for data-quality issues, unusual numerical patterns, and suspicious records, then export evidence for human review.

### Included
- Synthetic demo dataset with deliberately planted issues, or upload your own CSV
- Data-quality profile: missingness, duplicates, data types, unique values
- IQR-based numeric outlier screening
- Isolation Forest multivariate anomaly detection
- Interactive visualizations
- Investigation findings and downloadable CSV reports

> **Important:** anomaly flags are leads for investigation, not proof of fraud, corruption, or incorrect data. The notebook does not establish intent or causation. Review all findings using domain knowledge.

### How to use
Run cells from top to bottom. To use your own file, set `USE_DEMO_DATA = False` in the configuration cell and upload your CSV when prompted.


In [ ]:
# 1. Install dependencies (run once if needed)
%pip -q install pandas numpy scikit-learn matplotlib seaborn plotly ipywidgets


In [ ]:
# 2. Imports and configuration
from pathlib import Path
from datetime import datetime, timezone
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, Markdown
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
USE_DEMO_DATA = True  # Change to False to upload your own CSV
OUTPUT_DIR = Path("ghost_in_the_data_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("GHOST IN THE DATA initialized.")
print("Output directory:", OUTPUT_DIR.resolve())


In [ ]:
# 3. Create a demo dataset with deliberately planted issues
def create_demo_data(seed=17, n=700):
    rng = np.random.default_rng(seed)
    departments = np.array(["Engineering", "Finance", "Sales", "Operations", "People"])
    df = pd.DataFrame({
        "record_id": np.arange(10001, 10001 + n),
        "department": rng.choice(departments, n, p=[.30, .15, .22, .23, .10]),
        "tenure_years": np.clip(rng.normal(4.8, 3.2, n), 0, 25).round(1),
        "monthly_salary": np.clip(rng.normal(62000, 18000, n), 18000, 160000).round(0),
        "monthly_overtime_hours": np.clip(rng.gamma(2.0, 5.0, n), 0, 60).round(1),
        "performance_score": np.clip(rng.normal(3.4, .65, n), 1, 5).round(2),
        "sick_days": rng.poisson(3.2, n),
    })
    anomaly_idx = rng.choice(df.index, size=max(8, n // 70), replace=False)
    half = len(anomaly_idx) // 2
    df.loc[anomaly_idx[:half], "monthly_salary"] *= 3.5
    df.loc[anomaly_idx[half:], "monthly_overtime_hours"] = rng.uniform(
        75, 120, size=len(anomaly_idx) - half
    )
    missing_idx = rng.choice(df.index, size=max(10, n // 35), replace=False)
    df.loc[missing_idx, "performance_score"] = np.nan
    # Add five exact duplicate rows and a category typo for demonstration.
    df = pd.concat([df, df.iloc[:5]], ignore_index=True)
    df.loc[12, "department"] = "Enginering"
    return df

if USE_DEMO_DATA:
    df = create_demo_data()
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise ValueError("No CSV file uploaded.")
        df = pd.read_csv(csv_files[0])
    except ImportError:
        csv_path = input("Enter the path to your CSV file: ").strip()
        df = pd.read_csv(csv_path)

if df.empty:
    raise ValueError("The dataset is empty.")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head(10))


In [ ]:
# 4. Executive overview and basic data-quality metrics
n_rows, n_cols = df.shape
missing_cells = int(df.isna().sum().sum())
duplicate_rows = int(df.duplicated().sum())
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

overview = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing cells", "Exact duplicate rows",
               "Numeric columns", "Categorical/text columns"],
    "Value": [n_rows, n_cols, missing_cells, duplicate_rows,
              len(numeric_cols), len(categorical_cols)]
})
display(overview)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=overview.iloc[:4], x="Metric", y="Value", ax=ax)
ax.set_title("Dataset at a glance")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# 5. Build a column-level data-quality profile
def build_quality_report(data):
    rows = []
    n = max(len(data), 1)
    for col in data.columns:
        s = data[col]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()),
            "missing_percent": round(float(s.isna().mean() * 100), 2),
            "unique_values": int(s.nunique(dropna=True)),
            "unique_percent": round(float(s.nunique(dropna=True) / n * 100), 2),
        })
    return pd.DataFrame(rows).sort_values(
        ["missing_count", "unique_values"], ascending=[False, True]
    )

quality = build_quality_report(df)
display(quality)

missing_plot = quality[quality["missing_count"] > 0].sort_values("missing_count", ascending=False)
if not missing_plot.empty:
    fig = px.bar(missing_plot, x="column", y="missing_count",
                 title="Missing values by column", text="missing_count")
    fig.show()
else:
    print("No missing values found.")


In [ ]:
# 6. Inspect duplicate records and possible categorical inconsistencies
duplicate_mask = df.duplicated(keep=False)
print("Rows belonging to a duplicate group:", int(duplicate_mask.sum()))
print("Exact duplicate rows after the first occurrence:", int(df.duplicated().sum()))
if duplicate_mask.any():
    display(df.loc[duplicate_mask].sort_values(list(df.columns)).head(30))

categorical_summary = []
for col in categorical_cols:
    counts = df[col].value_counts(dropna=False)
    categorical_summary.append({
        "column": col,
        "unique_categories": int(df[col].nunique(dropna=True)),
        "most_common_value": str(counts.index[0]) if len(counts) else None,
        "most_common_count": int(counts.iloc[0]) if len(counts) else 0,
        "sample_categories": ", ".join(map(str, df[col].dropna().unique()[:12])),
    })
categorical_report = pd.DataFrame(categorical_summary)
display(categorical_report if not categorical_report.empty else pd.DataFrame({"note":["No categorical columns detected."]}))
print("Note: typo and semantic inconsistency detection requires explicit rules or domain knowledge.")


In [ ]:
# 7. Numeric distributions and IQR-based potential outliers
def iqr_outlier_report(data, columns):
    findings = []
    for col in columns:
        values = pd.to_numeric(data[col], errors="coerce")
        clean = values.dropna()
        if len(clean) < 8:
            continue
        q1, q3 = clean.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (values < lower) | (values > upper)
        findings.append({
            "column": col, "q1": q1, "q3": q3, "iqr": iqr,
            "lower_fence": lower, "upper_fence": upper,
            "potential_outlier_count": int(mask.sum()),
            "potential_outlier_percent": round(float(mask.mean() * 100), 2),
        })
    return pd.DataFrame(findings)

iqr_report = iqr_outlier_report(df, numeric_cols)
display(iqr_report if not iqr_report.empty else pd.DataFrame({"note":["No numeric columns had enough variation for IQR screening."]}))

plot_cols = numeric_cols[:6]
if plot_cols:
    df[plot_cols].hist(figsize=(14, 8), bins=30, edgecolor="black")
    plt.suptitle("Numeric feature distributions", y=1.02, fontsize=15)
    plt.tight_layout()
    plt.show()


In [ ]:
# 8. Multivariate anomaly detection with Isolation Forest
# Missing values are imputed for model input only; the original df remains unchanged.
CONTAMINATION = 0.05  # Tune this to your use case; it is not a known ground-truth rate.

def detect_anomalies(data, contamination=0.05):
    result = data.copy()
    cols = result.select_dtypes(include=np.number).columns.tolist()
    if len(cols) == 0 or len(result) < 10:
        result["anomaly_score"] = np.nan
        result["anomaly_flag"] = "Not evaluated"
        return result, []

    X = result[cols].replace([np.inf, -np.inf], np.nan)
    usable = [c for c in X.columns if not X[c].isna().all()]
    if not usable:
        result["anomaly_score"] = np.nan
        result["anomaly_flag"] = "Not evaluated"
        return result, []

    X_imp = SimpleImputer(strategy="median").fit_transform(X[usable])
    X_scaled = StandardScaler().fit_transform(X_imp)
    model = IsolationForest(
        n_estimators=250, contamination=contamination,
        random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_scaled)
    result["anomaly_score"] = -model.decision_function(X_scaled)
    result["anomaly_flag"] = np.where(
        model.predict(X_scaled) == -1, "Investigate", "Within expected pattern"
    )
    return result, usable

analyzed, features_used = detect_anomalies(df, CONTAMINATION)
flagged = analyzed[analyzed["anomaly_flag"] == "Investigate"].sort_values("anomaly_score", ascending=False)

print(f"Numeric features used: {len(features_used)}")
print(f"Rows flagged for review: {len(flagged):,} / {len(df):,} ({len(flagged)/max(len(df),1)*100:.2f}%)")
display(flagged.head(20))


In [ ]:
# 9. Visualize anomaly scores and two selected numeric features
if analyzed["anomaly_score"].notna().any():
    fig = px.histogram(
        analyzed, x="anomaly_score", color="anomaly_flag", nbins=40,
        title="Isolation Forest anomaly score distribution",
        labels={"anomaly_score": "Anomaly score (higher = more unusual)"}
    )
    fig.show()

    if len(features_used) >= 2:
        x_col, y_col = features_used[0], features_used[1]
        fig = px.scatter(
            analyzed, x=x_col, y=y_col, color="anomaly_flag",
            hover_data=list(df.columns[:min(5, len(df.columns))]),
            title=f"Anomaly candidates: {x_col} vs {y_col}"
        )
        fig.show()
else:
    print("Anomaly detection did not run. Check numeric columns and row count.")


In [ ]:
# 10. Combine findings into an investigation table
def build_issue_table(data, analyzed_data, quality_data, iqr_data):
    findings = []
    n = max(len(data), 1)

    dup_count = int(data.duplicated().sum())
    if dup_count:
        findings.append({
            "issue_type": "Exact duplicate rows", "severity": "Medium",
            "count": dup_count,
            "evidence": f"{dup_count} row(s) exactly duplicate a previous row.",
            "recommended_action": "Verify whether repeated records are valid before removal."
        })

    for _, row in quality_data[quality_data["missing_count"] > 0].iterrows():
        findings.append({
            "issue_type": "Missing values", "severity": "Medium" if row["missing_percent"] >= 5 else "Low",
            "count": int(row["missing_count"]),
            "evidence": f"Column '{row['column']}' has {row['missing_count']} missing value(s) ({row['missing_percent']}%).",
            "recommended_action": "Investigate the source and choose a context-appropriate handling strategy."
        })

    if not iqr_data.empty:
        for _, row in iqr_data[iqr_data["potential_outlier_count"] > 0].iterrows():
            findings.append({
                "issue_type": "Potential numeric outliers", "severity": "Review",
                "count": int(row["potential_outlier_count"]),
                "evidence": f"'{row['column']}' has {row['potential_outlier_count']} value(s) outside IQR fences.",
                "recommended_action": "Inspect source records and domain constraints; outliers can be valid."
            })

    if "anomaly_flag" in analyzed_data.columns:
        anomaly_count = int((analyzed_data["anomaly_flag"] == "Investigate").sum())
        findings.append({
            "issue_type": "Multivariate anomaly candidates", "severity": "Review",
            "count": anomaly_count,
            "evidence": f"Isolation Forest flagged {anomaly_count} row(s) based on combined numeric patterns.",
            "recommended_action": "Review the records with domain knowledge; a flag is not proof of error."
        })

    if not findings:
        findings.append({
            "issue_type": "No basic issues detected", "severity": "Info", "count": 0,
            "evidence": "No issues were reported by the current checks.",
            "recommended_action": "No flags does not guarantee the dataset is error-free."
        })
    return pd.DataFrame(findings)

issues = build_issue_table(df, analyzed, quality, iqr_report)
display(issues)


In [ ]:
# 11. Optional: compare dataset versions to find changed records
# Upload/define a second DataFrame as df_v2 to use this cell.
def compare_dataset_versions(df_v1, df_v2, key_column=None):
    if key_column and key_column in df_v1.columns and key_column in df_v2.columns:
        left = df_v1.set_index(key_column, drop=False)
        right = df_v2.set_index(key_column, drop=False)
        common = left.index.intersection(right.index)
        rows = []
        for key in common:
            for col in sorted(set(left.columns).intersection(right.columns)):
                if col == key_column:
                    continue
                a, b = left.loc[key, col], right.loc[key, col]
                if not ((pd.isna(a) and pd.isna(b)) or a == b):
                    rows.append({"record_key": key, "column": col, "old_value": a, "new_value": b})
        return pd.DataFrame(rows)
    # Fallback: compare common columns row-by-row up to the shorter dataset length.
    common_cols = sorted(set(df_v1.columns).intersection(df_v2.columns))
    rows = []
    for i in range(min(len(df_v1), len(df_v2))):
        for col in common_cols:
            a, b = df_v1.iloc[i][col], df_v2.iloc[i][col]
            if not ((pd.isna(a) and pd.isna(b)) or a == b):
                rows.append({"row_number": i + 1, "column": col, "old_value": a, "new_value": b})
    return pd.DataFrame(rows)

print("Version comparison helper is ready.")
print("To use: df_v2 = pd.read_csv('new_version.csv'); changes = compare_dataset_versions(df, df_v2, key_column='record_id')")


In [ ]:
# 12. Export outputs as CSV files
# The source row number links each output row back to the input order.
report_df = analyzed.copy()
report_df.insert(0, "source_row_number", np.arange(1, len(report_df) + 1))

quality_path = OUTPUT_DIR / "data_quality_profile.csv"
issues_path = OUTPUT_DIR / "investigation_findings.csv"
analyzed_path = OUTPUT_DIR / "analyzed_dataset.csv"
iqr_path = OUTPUT_DIR / "iqr_outlier_report.csv"

quality.to_csv(quality_path, index=False)
issues.to_csv(issues_path, index=False)
report_df.to_csv(analyzed_path, index=False)
iqr_report.to_csv(iqr_path, index=False)

print("Saved output files:")
for path in [quality_path, issues_path, analyzed_path, iqr_path]:
    print(f" - {path.resolve()} ({path.stat().st_size:,} bytes)")


In [ ]:
# 13. Final executive summary
flagged_count = int((analyzed["anomaly_flag"] == "Investigate").sum()) if "anomaly_flag" in analyzed else 0
summary = f'''
## GHOST IN THE DATA — Investigation Summary

- **Dataset size:** {len(df):,} rows × {df.shape[1]:,} columns
- **Missing cells:** {int(df.isna().sum().sum()):,}
- **Exact duplicate rows:** {int(df.duplicated().sum()):,}
- **Potential IQR outlier checks:** {len(iqr_report):,} numeric columns evaluated
- **Isolation Forest flags:** {flagged_count:,} row(s) marked for review
- **Generated at:** {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}

### Interpretation
These automated checks identify patterns worth investigating. They do **not** prove fraud, intent, data corruption, or root cause. Validate findings against source systems, domain rules, and data owners before changing records.
'''
display(Markdown(summary))


## Suggested GitHub next steps

1. Add a `README.md` with screenshots, setup instructions, methodology, limitations, and sample outputs.
2. Add tests for duplicate detection, missing-value reporting, and anomaly output shape.
3. Add configurable domain-specific validation rules (for example, valid category lists and numeric bounds).
4. Add a version comparison workflow using a stable record ID.
5. Never publish private or sensitive datasets; use the synthetic demo or properly licensed public data.

**Suggested repository name:** `ghost-in-the-data`  
**Tagline:** *Every dataset leaves clues. Find them.*
